In [ ]:
import re
import pandas as pd
import numpy as np


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files

In [ ]:
# dataset με προτασιοπιημένα κείμενα
df = pd.read_csv('/my_path/LeMonde_RussoUkrainian_Sents.csv', encoding='utf-8')


In [ ]:
df.columns

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 50)

In [ ]:
df.head(2)

In [ ]:
df.shape

In [ ]:
#πόσες προτάσεις ανα χρονική περίοδο?
df['Time_period'].value_counts()

In [ ]:
df.columns

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 50)

## Encode text

In [ ]:
# το torch είναι η βασική βιβλιοθήκη του PyTorch και χρησιμοποιείται για neural networks, embeddings και υπολογισμούς με GPU
# το μοντέλο sentence-transformers που εισάγουμε παρκαάτω αξιοποιεί εσωτερικά το PyTorch

import torch


In [ ]:
# ελέγχω αν τρέχω σε GPU or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
from sentence_transformers import SentenceTransformer, util

##### Τσέκαρω στο https://sbert.net/ για documentation

In [ ]:
model = SentenceTransformer('all-mpnet-base-v2')  #πολύ καλό μοντέλο για clustering, check https://huggingface.co/sentence-transformers/all-mpnet-base-v2

In [ ]:
model

In [ ]:
#sanity check
emb = model.encode(["hello world", "this is a test"], normalize_embeddings=True)
emb.shape

In [ ]:
# μετατέπω τις καθαρές προτάσεις ή τα καθαρα κείμενα σε λίστα
corpus = list(df['sents_clean'])  # sentences to embed

In [ ]:
df.columns

In [ ]:
#φτιάχνω το corpus με το οποίο θα προχωρήσω
corpus_df = pd.DataFrame({'Text_ID': df['Text_id'],'Sent_ID': df['Sent_id'], 'Article_url': df['Article_url'], 'Title': df['title'], 'date': df['date'], 'Sentences': corpus, 'Period': df['Time_period'],'site': df['site'],
                          'Country': df['Country'], 'Orientation': df['Orientation']})


In [ ]:
corpus_df.head(2)

In [ ]:
corpus_df.shape

### Vectorization

In [ ]:
# περνάω το corpus στο μοντέλο για να δημιουργήσω διανύσμaτα (vectors)
# υπολογίζω embeddings για κάθε πρόταση (ένα διάνυσμα ανά πρόταση)

corpus_embeddings = model.encode(corpus_df['Sentences'].tolist(), show_progress_bar=True)


In [ ]:
# ελέγχω τα διανύσματα
corpus_embeddings

In [ ]:
corpus_embeddings.shape

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 50)

In [ ]:
# αποθηκεύω τα embeddings σε δική τους στήλη στο dataframe
corpus_df['Embeddings'] = list(corpus_embeddings)  # Store embeddings as a list in the DataFrame

In [ ]:
#check
corpus_df.head()

## Clustering

### 1. Dimensionality Reduction με Umap
Ελέγχω το documentation εδώ https://umap-learn.readthedocs.io/en/latest/

In [ ]:
# εισαγώ το umap για μείωση διστάσεων
import umap

Τα embeddings είναι ουσιαστικά πίνακες αριθμών. Για να μειώσουμε τις διαστάσεις και να προχωρήσουμε στο clustering θα χρησιμοποιήσουμε και τη βιβλιοθήκη numPy που βοηθά να τους επεξεργαστουμε γρηγορα και αποδοτικά

In [ ]:
# εισάγω τη βιβλιοθήκη numpy
import numpy as np

In [ ]:
# δοκιμάζω διαφορετικές τιμές n_neighbors για να δω πώς αλλάζει η δομή του χώρου

neighbor_values = [10, 15, 20, 30, 50, 100]  # διαφορετικά μεγέθη γειτονιάς
avg_distances = []  # λίστα όπου θα αποθηκεύσω τη μέση απόσταση των σημείων

# eπαναλαμβάνω τη διαδικασία για κάθε τιμή n_neighbors
for n in neighbor_values:

    # δημιουργώ προσωρινό μοντέλο UMAP
    umap_test = umap.UMAP(
        n_neighbors=n,      # πόσους "γείτονες" κοιτάζει κάθε σημείο
        n_components=5,     # πόσες διαστάσεις θα κρατήσει το UMAP
        metric="cosine",    # =μέτρο ομοιότητας για embeddings κειμένου
        random_state=42     # για reproducible αποτελέσματα
    )

    # εφαρμόζω UMAP στα embeddings μου
    embedding_test = umap_test.fit_transform(corpus_embeddings)

    # υπολογίζω τη μέση απόσταση των σημείων από το κέντρο του χώρου
    avg_dist = np.mean(
        np.linalg.norm(
            embedding_test - embedding_test.mean(axis=0),
            axis=1
        )
    )

    # αποθηκεύω τη μέση απόσταση για plotting
    avg_distances.append(avg_dist)

In [ ]:
# plot τα αποτελεσαμτα
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(neighbor_values, avg_distances, marker="o", linestyle="-")
plt.xlabel("n_neighbors")
plt.ylabel("Average Distance in UMAP Space")
plt.title("Optimal n_neighbors Selection")
plt.show()

Τι δείχνει το γράφημα: αν επιλέξω μικρό n_neighbors, π.χ. 10 το UMAP κοιτάζει μόνο πολύ κοντινούς γείτονες, άρα εστιάζει σε μικρές τοπικές λεπτομέρειες, δημιουργεί πιο "σφιχτά" clusters αλλά μπορεί να "σπάσει" τον χώρο. Μεγάλο n_neighbors, π.χ. 100, κοιτάζει πολύ περισσότερα σημεία γύρω από κάθε observation, άρα βλέπει τη "γενική εικόνα", ο χώρος γίνεται πιο ομαλός
αλλά χάνεται η τοπική λεπτομέρεια. Ο άξονας Υ δείχνει πόσο μακριά βρίσκονται κατά μέσο όρο τα σημεία (στην περίπωσή μας οι προτάσεις) μεταξύ τους μετά τη μείωση διαστάσεων. Επομένως το  σημείο καμπής (elbow) δείχνει ότι από εκεί και μετά το να αυξήσουμε τον αριθμό n_neighbors δεν αλλάζει πολύ τη δομή γιατί δεν αλλάζει η απόσταση ανάμεσα στα σημεία. Το γράφημά μας υποδεικνύει μια τιμή ανάμεσα στο 30 και το 40.

In [ ]:
# δημιουργώ το τελικό μοντέλο UMAP με τις παραμέτρους που θέλω
umap_model = umap.UMAP(
    n_neighbors=35,       # ορίζει πόσα κοντινά σημεία λαμβάνονται υπόψη για τον ορισμό της τοπικής δομής- μεγεθος 'γειτονιάς' (παράθυρο χώρου)
    n_components=5,       # μείωση σε 5 διαστάσεις
    metric='cosine',      # αποστασή γωνίας συνημίτονου (cosine distance)--> μετρά πόσο παρόμοια είναι δύο embeddings με βάση τη γωνία τους και την κατεύθυνση των διανυσμάτων τους
    random_state=42       # για reproducibility
)

# τρέχω το UMAP πάνω στα sentence embeddings μου
umap_embeddings = umap_model.fit_transform(corpus_embeddings)  # corpus_embeddings είναι τα input data

In [ ]:
# αποθηκεύω τις UMAP συντεταγμένες σε δική τους στήλη στο df μου
corpus_df['umap_embeddings'] = list(umap_embeddings)

### 2. Clustering - HDBSCAN
ελέγχω το documentation εδώ https://hdbscan.readthedocs.io/en/latest/how_hdbscan_works.html

In [ ]:
# εισάγω τη βιβλιοθήκη hdbscan για clustering
import hdbscan

In [ ]:
# δοκιμάζω διαφορετικές τιμές min_cluster_size για να δω πόσα clusters βγαίνουν
min_cluster_sizes = [20, 30, 40, 50, 100, 150, 200, 250, 300]

for size in min_cluster_sizes:

    # HDBSCAN μοντέλο με συγκεκριμένο min_cluster_size
    cluster_model = hdbscan.HDBSCAN(
        min_cluster_size = size,
        metric='euclidean',    # ευκλείδια απόσταση μεταξύ σημείων (προτάσεων/κειμένων) στο UMAP space
        cluster_selection_method='eom' #eom=Excess of Mass method, μέθοδος επιλογής clusters (το eom συνήθως δημιουργεί πιο σταθερά clusters)
    )

    # εφαρμόζω το clustering στα UMAP embeddings μου
    cluster = cluster_model.fit(umap_embeddings)

    #τυπώνω πόσα διαφορετικά clusters βρέθηκαν σε κάθε τιμή min_cluster_size
    print(f"Min Cluster Size: {size}, Number of Clusters: {len(np.unique(cluster.labels_))}")

#### Πώς επιλέγουμε το min_cluster_size;

Το min_cluster_size ορίζει πόσα σημεία χρειάζονται
τουλάχιστον για να δημιουργηθεί ένα cluster.

Μικρές τιμές:
- δημιουργούν πολλά μικρά clusters, πιάνουν περισσότερες λεπτομέρειες αλλά μπορεί να δημιουργήσουν "θόρυβο"

Μεγάλες τιμές:
- δημιουργούν λιγότερα και μεγαλύτερα clusters, δίνουν πιο γενικές θεματικές ομάδες αλλά μπορεί να χαθούν μικρότερα θέματα.

Συνήθως ψάχνουμε clusters που έχουν νοηματική συνοχή, σχετική σταθερότητα (όχι υπερβολικά πολλά ή υπερβολικά λίγα clusters)


In [ ]:
# δημιουργώ το τελικό μοντέλο HDBSCAN για clustering

cluster_model = hdbscan.HDBSCAN(
    min_cluster_size=40,            # ελάχιστο πλήθος σημείων για να θεωρούνται cluster ('γειτονιά')
    min_samples=2,                  # πόσα κοντινά σημεία απαιτούνται ώστε ένα σημείο να θεωρηθεί 'core' - κεντρικό σημείο cluster (μεγαλύτερες τιμές = πιο αυστηρό clustering)
    metric='euclidean',             # μέτρο απόστασης μεταξύ των σημείων-εδώ χρησιμοπιοούμε ευκλείδια απόσταση ανάμεσα στα σημεία στο UMAP space
    cluster_selection_method='eom'   # επιλογή clusters με τη μέθοδο Excess of Mass
)

# εφαρμόζω το τελικό clustering στο UMAP space
cluster = cluster_model.fit(umap_embeddings)


In [ ]:
# αποθηκεύω τα παραγόμενα cluster σε δική τους στήλη - το hdbscan έχει δώσει αριθμους σε κάθε cluster
corpus_df['Cluster'] = cluster_model.labels_

In [ ]:
# παίρνω τα labels που έδωσε το HDBSCAN --> Κάθε σημείο έχει έναν αριθμό cluster ή -1 αν θεωρήθηκε outlier/noise
labels = cluster.labels_

# υπολογίζω πόσα clusters έχουν προκύψει χωρίς να μετράω τα outliers (-1)
num_clusters = len(set(labels)) - (1 if -1 in labels else 0)

# τυπώνω τον συνολικό αριθμό clusters
print(f"Number of clusters (excluding outliers): {num_clusters}")

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 50)

In [ ]:
# εισάγω το έργαλείο για μέτρηση συχνοτήτων
from collections import Counter

# μετράω πόσα σημεία ανήκουν σε κάθε cluster
cluster_counts = Counter(labels)

# εμφανίζω το μέγεθος κάθε cluster
print("Cluster sizes:")
for cluster_id, count in cluster_counts.items():
    if cluster_id != -1: #αγνοώ τα outliers (-1)
        print(f"Cluster {cluster_id}: {count} points") # τυπώνω πόσα σημεία (προτάσεις/κείμενα) έχει κάθε cluster

In [ ]:
corpus_df.tail()

In [ ]:
# τσεκάρω πόσες προτάσεις ταξινομήθηκαν ως outliers (-1)
outliers = corpus_df[corpus_df['Cluster'] == -1]
print(f"Number of sentences in the -1 cluster: {len(outliers)}")

## Optional: αν θέλω να "σώσω" κάποια outliers:
Τα outliers είναι σημεία που το HDBSCAN δεν μπόρεσε να εντάξει με ασφάλεια σε cluster. Ελέγχω αν μερικά outliers βρίσκονται πολύ κοντά σε υπάρχοντα clusters στο UMAP space.

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [ ]:
# χωρίζω τα σημεία σε:

# 1. κανονικά clustered points
core_mask = corpus_df['Cluster'] != -1  # true για σημεία που ανήκουν ήδη σε cluster
# 2. outliers
outlier_mask = ~core_mask    # true για σημεία που είναι outliers (-1)

# παίρνω τα labels των clustered σημείων, δηλαδή σε ποιο cluster ανήκει το κάθε σημείο
core_labels = corpus_df.loc[core_mask, 'Cluster'].to_numpy()

# δημιουργώ πίνακα με τα UMAP embeddings των clustered σημείων
core_embeddings = np.vstack(corpus_df.loc[core_mask, 'umap_embeddings'])

# δημιουργώ πίνακα με τα UMAP embeddings των outliers
outlier_embeddings = np.vstack(corpus_df.loc[outlier_mask, 'umap_embeddings'])



Υπολογίζω πόσο κοντά βρίσκονται συνήθως τα σημεία μέσα στα clusters


In [ ]:
# θα χρησιμοποιήσω τις αποστάσεις για να αποφασίσω αν κάποιο outlier είναι αρκετά κοντά ώστε να μπει σε cluster

# δημιουργώ μοντέλο k-Nearest Neighbors πάνω στα core σημεία (τα σημεία που ήδη ανήκουν σε clusters) και βρισκω ποιος είναι ο πιο κοντινός τους γείτονας.
nn_core = NearestNeighbors(n_neighbors=2, metric='cosine').fit(core_embeddings) # n_neighbors=2 γιατί ο 1ος γείτονας είναι το ίδιο το σημείο, ο 2ος είναι ο πιο κοντινός πραγματικός γείτονας


# υπολογίζω τις αποστάσεις των core σημείων από τους κοντινότερους γείτονές τους
dists_core, _ = nn_core.kneighbors(core_embeddings)

# κρατάω μόνο την απόσταση προς τον πιο κοντινό πραγματικό γείτονα (αγνοώ το ίδιο το σημείο)
core_nn1 = dists_core[:, 1]  #πόσο κοντά είναι συνήθως τα σημεία μέσα στα clusters;

# κάνω τους υπολογισμους με threshold το 97ο ποσοστημόριο, δηλαδή το 97% των core σημείων έχουν μικρότερη απόσταση από αυτήν την τιμή
thr = np.quantile(core_nn1, 0.97)

# τυπώνω το threshold
print(f"Calibrated threshold (97th pct of core NN distances): {thr:.6f}")


Το 97% των σημείων που ανήκουν ήδη σε clusters έχουν απόσταση μικρότερη ή ίση με 0.000048 από τον κοντινότερο γείτονά τους. Άρα αν ένα outlier βρίσκεται τόσο κοντά σε κάποιο cluster, τότε πιθανόν ανήκει σε αυτό και μπορούμε να το αναθέσουμε εκεί.

In [ ]:
# εκπαιδεύω kNN πάνω στα clustered (non-outlier) embeddings
nn = NearestNeighbors(n_neighbors=1, metric='cosine').fit(core_embeddings)

# βρίσκω για κάθε outlier τον κοντινότερο core (clustered) γείτονα
dists_out, idx_out = nn.kneighbors(outlier_embeddings) # dists_out: απόσταση κάθε outlier προς τον κοντινότερο core, idx_out: index του κοντινότερου core σημείου

# μετατρέπουμε το "nearest core point" σε cluster id μέσω των core_labels
nearest_clusters = core_labels[idx_out.ravel()] #ποιο cluster έχει αυτό το κοντινό core σημείο;

# αν ο outlier είναι αρκετά κοντά (<= thr) τον βάζουμε στο cluster του κοντινού core αλλιώς μένει -1 (outlier)
accepted_clusters = np.where(dists_out.ravel() <= thr, nearest_clusters, -1) #αν το outlier είναι ΠΟΛΥ κοντά σε cluster → το βάζω εκεί αν όχι → μένει outlier (-1)


In [ ]:
# ενημερώνω την στήλη των cluster
# γράφω τις νέες αναθέσεις ΜΟΝΟ στις γραμμές που ήταν outliers--> τα core σημεία μένουν προσωρινά NaN στο Updated_Cluster
corpus_df.loc[outlier_mask, 'Updated_Cluster'] = accepted_clusters

In [ ]:
# summary
n_outliers = outlier_embeddings.shape[0] # πλήθος outliers που εξετάσαμε
n_assigned = (accepted_clusters != -1).sum() # πόσοι από αυτούς απέκτησαν cluster (δηλ. δεν έμειναν -1)

# check πόσοι outliers ανατέθηκαν
print(f"Assigned {n_assigned}/{n_outliers} outliers "
      f"({n_assigned/n_outliers:.1%}) using threshold {thr:.6f}")

In [ ]:
corpus_df.tail()

In [ ]:
# γεμίζω τα NaN: όπου δεν ορίστηκε Updated_Cluster κρατάμε το label του αρχικού Cluster
corpus_df['Updated_Cluster'] = corpus_df['Updated_Cluster'].fillna(corpus_df['Cluster'])

# μετατρέπω το Updated_Cluster σε int (clusters ως ακέραιοι)
corpus_df['Updated_Cluster'] = corpus_df['Updated_Cluster'].astype(int)

In [ ]:
corpus_df.tail(2)

In [ ]:
# πόσες προτάσεις παραμένουν outliers μετά την ενημέρωση
updated_outliers_count = (corpus_df['Updated_Cluster'] == -1).sum()
print(f"Remaining sentences in the -1 cluster: {updated_outliers_count}")


In [ ]:
# πόσες ήταν αρχικά outliers (με βάση το αρχικό Cluster) και παραμένουν ανένταχτες;
remaining_outliers = corpus_df[corpus_df['Cluster'] == -1]
print(f"Remaining sentences originally marked as outliers: {len(remaining_outliers)}")

# πόσες παραμένουν outliers μετά το reassignment;
still_unassigned = corpus_df[corpus_df['Updated_Cluster'] == -1]
print(f"Sentences still unassigned after update: {len(still_unassigned)}")

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 200)

In [ ]:
corpus_df.head()

### Visualize
Ομαδοποιήσαμε τις προτάσεις σε ομάδες με παρόμοιο νόημα. Για να αναπαραστήσουμε οπτικά την ομαδοποίηση και να ελεγξουμε τα αποτελεσματά, μειώνουμε τα αρχικά πολυδιάστατα embeddings σε δύο διαστάσεις με τη μέθοδο UMAP.
Κάθε σημείο στο γράφημα αντιστοιχεί σε μία πρόταση. Τα σημεία με ετικέτα -1 αντιπροσωπεύουν τα outliers (δηλαδή προτάσεις που δεν ανατέθηκαν σε κάποιο cluster) και εμφανίζονται με γκρι χρώμα. Τα υπόλοιπα σημεία χρωματίζονται ανάλογα με το cluster στο οποίο έχουν ενταχθεί.
*Σημείωση: οι αριθμητικές ετικέτες των clusters αποτελούν αναγνωριστικά που παράγονται από τον αλγόριθμο και δεν έχουν από μόνες τους ερμηνευτικό νόημα*.


In [ ]:
# χαμηλώνω τα embeddings σε 2 διαστασεις για να δω τα clusters και τα outliers (-1)
umap_data = umap.UMAP(n_neighbors=35, n_components=2, # 2 διαστάσεις για visualization
                      min_dist=0.0, metric='cosine').fit_transform(corpus_embeddings)


In [ ]:
# φτιάχνω dataframe με τις 2D συντεταγμένες
result = pd.DataFrame(umap_data, columns=['x', 'y'])  # x,y coords στο UMAP plane
result['labels'] = corpus_df['Updated_Cluster']  # labels = Updated_Cluster (μετά το reassignment)


In [ ]:
# scatter plot
fig, ax = plt.subplots(figsize=(12, 12))  # μέγεθος γραφήματος

# χωρίζω outliers και non-outliers
outliers = result[result['labels'] == -1] # σημεία χωρίς cluster
clustered = result[result['labels'] != -1] # σημεία που ανήκουν σε cluster

#  outliers σε γκρι χρώμα
plt.scatter(outliers['x'], outliers['y'], color='#BDBDBD', s=8, alpha=0.5, label='Outliers')

# plot τα clustered σημεία με διαφορετικό χρώμα ανά cluster id
scatter = plt.scatter(
    clustered['x'], clustered['y'], c=clustered['labels'], cmap='hsv_r', s=2, alpha=0.7
)

# colorbar και labels
plt.colorbar(scatter, label='Cluster Labels') # βάζω τα χρώματα βάση του label
plt.title("UMAP Projection with HDBSCAN Clusters")
plt.xlabel("UMAP Dimension 1") # άξονας x
plt.ylabel("UMAP Dimension 2") # άξονας y
plt.legend()
plt.show()

Το UMAP projection είναι 2D οπτικοποίηση.Δηλαδή βλέπουμε μια απλοποιημένη εικόνα
της πραγματικής δομής των embeddings. H εικόνα δείχνει ότι κάποια clusters είναι πιο "σφιχτά"και απομονωμένα, ενώ άλλα αλληλεπικαλύπτονται περισσότερο, κάτι που δείχνει ότι ορισμένα θέματα είναι πιο ξεκάθαρα και άλλα πιο μικτά ή αμφίσημα.



In [ ]:
corpus_df.head(1)

# Find Topics in clusters
Εξάγουμε keywords ώστε να καταλάβουμε "τι λέει" κάθε cluster --> Μέθοδος: class-based TF-IDF (c-TF-IDF) πάνω σε concatenated κείμενο ανά cluster

In [ ]:
# ομαδοποιώ τις προτάσεις ανά cluster και τις ενώνω σε ένα ενιαίο document
docs_per_cluster = corpus_df.groupby(['Updated_Cluster'], as_index=False).agg({'Sentences': ', '.join}) # κάθε row εδώ = 1 cluster με ένα μεγάλο κείμενο


In [ ]:
docs_per_cluster.shape

In [ ]:
pd.set_option('display.width', 50)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', 2000)

#### Ας δουμε πως είναι ένα cluster...

In [ ]:
docs_per_cluster.head(1)

In [ ]:
# CountVectorizer: μετατρέπει κείμενα σε πίνακα συχνοτήτων n-grams

from sklearn.feature_extraction.text import CountVectorizer
import numpy as np


In [ ]:
# function για υπολογισμό class-based tf-idf
# c-TF-IDF: terms που είναι συχνοί σε ένα cluster αλλά σχετικά σπάνιοι στο σύνολο του corpus παίρνουν υψηλό score

def c_tf_idf(documents, m, ngram_range=(1, 2)):
    # m = συνολικός αριθμός αρχικών documents (π.χ. sentences πριν το aggregation σε clusters)
    # ngram_range = εύρος n-grams (π.χ. 1 για unigrams και 2 για bigrams)

    count = CountVectorizer(ngram_range=ngram_range, stop_words="english").fit(documents)  # δημιουργία vocabulary
    t = count.transform(documents).toarray()  # counts matrix (clusters x terms)

    w = t.sum(axis=1)  # συνολικός αριθμός όρων ανά cluster-document

    # term frequency ανά cluster (κανονικοποιημένο ως προς το μέγεθος του cluster)
    tf = np.divide(t.T, w, out=np.zeros_like(t.T, dtype=float), where=w != 0)

    sum_t = t.sum(axis=0)  # συνολική συχνότητα κάθε term σε όλα τα clusters

    # idf: 'τιμωρεί' terms που έχουν υψηλή συνολική συχνότητα στο corpus
    idf = np.log(np.divide(m, sum_t, out=np.zeros_like(sum_t, dtype=float), where=sum_t != 0)).reshape(-1, 1)

    tf_idf = np.multiply(tf, idf)  # c-TF-IDF scores (terms x clusters)

    return tf_idf, count  # επιστρέφει scores και το vectorizer

In [ ]:
# υπολογίζω c-TF-IDF για τα cluster-documents
tf_idf, count = c_tf_idf(docs_per_cluster['Sentences'].values, # κείμενο ανά cluster
                         m=len(corpus_df)) # len= συνολικός αριθμός αρχικών sentences στο corpus

Για τη δημιουργία μιας αναπαράστασης θεμάτων (topic representation), επιλέγω 40 λέξεις ή φράσεις (n-grams) με την υψηλότερη τιμή c-tf-idf για κάθε cluster. Η τιμή c-TF-IDF αποτυπώνει πόσο χαρακτηριστικός είναι ένας όρος για ένα συγκεκριμένο cluster σε σχέση με το συνολικό corpus. Όσο υψηλότερη είναι η τιμή, τόσο περισσότερο ο όρος θεωρείται αντιπροσωπευτικός του συγκεκριμένου θέματος --> άρα κάθε cluster συνοψίζεται από ένα σύνολο λέξεων-κλειδιών που αποτυπώνουν το κυρίαρχο σημασιολογικό και ρητορικό του περιεχόμενο.

In [ ]:
# ορίζω και παίρνω τα top-N words ανά cluster (με βάση το c-TF-IDF score)

def extract_top_n_words_per_cluster(tf_idf, count, docs_per_cluster, n=40):
    words = count.get_feature_names_out() # όλα τα terms/ του vocbulary
    labels = list(docs_per_cluster['Updated_Cluster'])  # # cluster ids (ένα ανά cluster-document)

    tf_idf_transposed = tf_idf.T # μετατροπή σε (clusters x terms)
    indices = tf_idf_transposed.argsort()[:, -n:] # indices των top-n terms (μεγαλύτερες τιμές c-TF-IDF) ανά cluster

    top_n_words = {
        label: [(words[j], tf_idf_transposed[i][j]) for j in indices[i]][::-1] # φθίνουσα ταξινόμηση--> από το υψηλότερο στο χαμηλότερο score
        for i, label in enumerate(labels)
    }
    return top_n_words

# τοπ keywords για κάθε cluster
top_keywords = extract_top_n_words_per_cluster(tf_idf, count, docs_per_cluster, n=40)

In [ ]:
# check
top_keywords

In [ ]:
# φτιάχνω df με μέγεθος cluster + top keywords

def extract_cluster_sizes(df, top_keywords):
    cluster_sizes = (df.groupby(['Updated_Cluster'])
                       .Sentences.count() # πόσες προτάσεις/κείμενα ανά cluster
                       .reset_index()
                       .rename({"Updated_Cluster": "Cluster", "Sentences": "Size"}, axis='columns')
                       .sort_values("Size", ascending=False))

     # Top_Keywords: βάζω μόνο τις λέξεις (όχι τα scores) διαχωρισμένες με κόμμα
    cluster_sizes['Top_Keywords'] = cluster_sizes['Cluster'].apply(lambda x: ', '.join([word[0] for word in top_keywords[x]]))
    return cluster_sizes


In [ ]:
# πίνακας cluster sizes + keywords
cluster_sizes = extract_cluster_sizes(corpus_df, top_keywords)

In [ ]:
pd.set_option('display.max_colwidth', None)
cluster_sizes.head(50)

In [ ]:
cluster_sizes.shape

In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 50)

In [ ]:
# από το corpus_df αφαιρώ την αρχικη στήλη Cluster για να μείνει μόνο το Updated_Cluster

corpus_df.drop('Cluster', axis=1, inplace=True)
corpus_df.head(1)

In [ ]:
# κανω Rename τη στήλη Cluster στο dataframe 'cluster_sizes' ώστε να ταιριάζει για το merge key
cluster_sizes.rename(columns={'Cluster': 'Updated_Cluster'}, inplace=True)


In [ ]:
# check
cluster_sizes

In [ ]:
# κάνω merge πίσω στο corpus_df για να έχει κάθε πρόταση το Size του cluster και τα Top_Keywords του
corpus_df = corpus_df.merge(cluster_sizes, on='Updated_Cluster', how='left')

In [ ]:
corpus_df.head()

Για να ελέγξω το περιεχόμενο κάνω ένα μικρό πινακα με το Cluster, το Size, τα Top_Keywords και ένα δείγμα προτάσεων από κάθε cluster

In [ ]:
cluster_review = (
    corpus_df[corpus_df['Updated_Cluster'] != -1]
    .groupby('Updated_Cluster')
    .apply(lambda x: pd.Series({
        'Size': len(x),
        'Top_Keywords': x['Top_Keywords'].iloc[0],
        'Sample_Sentences': x['Sentences'].sample(
            min(5, len(x)),
            random_state=42
        ).tolist()
    }))
    .reset_index()
)



In [ ]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 450)

In [ ]:
cluster_review.head()

In [ ]:
# για να δω πιο καθαρά ένα cluster:

cluster_id = 3

print("Top keywords:")
print(corpus_df[corpus_df['Updated_Cluster'] == cluster_id]['Top_Keywords'].iloc[0])

print("\nSample sentences:")
for sent in corpus_df[corpus_df['Updated_Cluster'] == cluster_id]['Sentences'].sample(20, random_state=42): #τυπώνω 20 προτασεις, αν θέλω αλλαξω το νούμερο
    print("-", sent)

Τώρα μπορούμε ελέγχοντας τα keywords και ένα δείγμα προτάσεων από τη στήλη 'Sentences' να δημιουργήσουμε τίτλάκια για τα clusters μας


In [ ]:
# αν θέλω να δουλέψω ένα συγκεκριμένο cluster

corpus_df.loc[
    corpus_df['Updated_Cluster'] == 4, # το αριθμητικό id tou cluster
    'Cluster_title'
] = "Gaza War as a Geopolitical Opportunity for Russia" #για παράδειγα

In [ ]:
# αν εχω καταλήξει σε ακρέτα τιτλάκια προτιμώ να κανω mapping:

#για παράδειγμα
cluster_titles = {
    0: "Climate Politics",
    1: "War and Economic Crisis",
    3: "Latin American Diplomacy",
    4: "Gaza, Ukraine, Russia
}


In [ ]:
corpus_df['Cluster_title'] = corpus_df['Updated_Cluster'].map(cluster_titles)

In [ ]:
corpus_df.head()

In [ ]:
#σώζω
corpus_df.to_csv('/my_path/LeΜonde_RussoUkrainian_Sents_EMBS_Clusters.csv', index=False)
